In [ ]:
# ================================================================
# STEERING SWEEP + ADAPTIVE PER-SAMPLE STEERING
# Llama-3.2-3B-Instruct | TriviaQA | Google Colab A100
#
# PART 1 — Steering Sweep
#   Apply CAA at every layer, measure factual accuracy
#   → Finds correction layer L_correct (expected: 24)
#   → Produces two-panel figure: AUROC curve + steering accuracy curve
#
# PART 2 — Adaptive Per-Sample Steering
#   Router: uses probe confidence at L*=13 to decide
#   how aggressively to steer at L_correct=24
#   → Compares: no steering | fixed CAA | adaptive steering
#
# Runtime: ~60 min on A100
# Requires: q4_layer_comparison cache from previous notebook
# ================================================================

In [ ]:
# ── CELL 1: Imports & config ─────────────────────────────────────
# !pip install transformers datasets accelerate scikit-learn matplotlib tqdm -q

import os, re, gc, random
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR     = '/content/drive/MyDrive/q4_layer_comparison'
SWEEP_DIR    = '/content/drive/MyDrive/steering_sweep'
os.makedirs(SWEEP_DIR, exist_ok=True)

DEVICE       = 'cuda'
MODEL_ID     = 'meta-llama/Llama-3.2-3B-Instruct'
NUM_SAMPLES  = 500
MAX_NEW_TOK  = 50
RANDOM_SEED  = 42
DETECT_LAYER = 13      # L* — confirmed detection layer
ALPHA        = 2.0     # CAA steering magnitude (tune if needed)

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print("Setup complete ✓")

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# It's recommended to store your Hugging Face token in Colab secrets under the name 'HF_TOKEN'
# You can access Colab secrets from the '🔑' icon in the left panel.
HF_TOKEN = userdata.get('HF_TOKEN')

# Log in to Hugging Face
login(token=HF_TOKEN)

In [ ]:
# ── CELL 2: Load model ───────────────────────────────────────────
print(f"Loading {MODEL_ID} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    output_hidden_states=True
)
model.eval()

NUM_LAYERS  = model.config.num_hidden_layers
HIDDEN_DIM  = model.config.hidden_size
print(f"Model loaded ✓  |  Layers: {NUM_LAYERS}  |  Hidden dim: {HIDDEN_DIM}")

In [ ]:
# ── CELL 3: Load cache + TriviaQA ───────────────────────────────
# Load hidden states from Q4 notebook (already extracted)
# Also reload TriviaQA so we can re-generate with steering injected

CACHE_FILE = f'{SAVE_DIR}/hidden_states_cache.pt'
assert os.path.exists(CACHE_FILE), \
    f"Cache not found at {CACHE_FILE}. Run Q4 notebook first."

cache      = torch.load(CACHE_FILE)
all_hidden = cache['hidden_states']   # list of [L+1, D] tensors
all_labels = np.array(cache['labels'])

H = torch.stack(all_hidden, dim=0).numpy()   # [N, L+1, D]
y = all_labels

print(f"Hidden states loaded ✓  |  Shape: {H.shape}")
print(f"Labels: {y.sum()} factual / {(1-y).sum()} hallucinated")

# Reload TriviaQA (same seed = same samples)
dataset = load_dataset('trivia_qa', 'rc.nocontext', split='validation')
samples = dataset.shuffle(seed=RANDOM_SEED).select(range(NUM_SAMPLES))

def normalize(s):
    return re.sub(r'\b(a|an|the)\b', ' ',
           re.sub(r'[^\w\s]', '', s.lower())).strip()

def token_f1(pred, gold_list):
    pred_toks = normalize(pred).split()
    best = 0.0
    for gold in gold_list:
        gold_toks = normalize(gold).split()
        common = set(pred_toks) & set(gold_toks)
        if not common:
            continue
        p = len(common) / len(pred_toks) if pred_toks else 0
        r = len(common) / len(gold_toks) if gold_toks else 0
        f1 = 2*p*r/(p+r) if (p+r) > 0 else 0
        best = max(best, f1)
    return best

PROMPT_TMPL = (
    "Answer the following question in one short phrase.\n"
    "Question: {q}\nAnswer:"
)

print("TriviaQA loaded ✓")

In [ ]:
# ── CELL 4: Compute CAA steering vector at each layer ────────────
# CAA (Contrastive Activation Addition):
#   steering_vec = mean(factual hidden states) - mean(hallu hidden states)
#   at a given layer.
# We compute one steering vector per layer using the cached hidden states.

print("Computing CAA steering vectors for all layers ...")

steering_vecs = {}   # layer_idx → steering vector [D]

for layer_idx in range(NUM_LAYERS + 1):
    H_layer   = H[:, layer_idx, :]          # [N, D]
    fact_mean = H_layer[y == 1].mean(axis=0)
    hall_mean = H_layer[y == 0].mean(axis=0)
    vec       = fact_mean - hall_mean
    # Normalize to unit norm
    vec       = vec / (np.linalg.norm(vec) + 1e-8)
    steering_vecs[layer_idx] = torch.tensor(vec, dtype=torch.float16).to(DEVICE)

print(f"Steering vectors computed ✓  ({len(steering_vecs)} layers)")

In [ ]:
# ── CELL 5: Fit probe at DETECT_LAYER for router ─────────────────
# This probe will be used in Part 2 (adaptive steering)
# to get per-sample hallucination confidence at L*=13

print(f"Fitting probe at detection layer L*={DETECT_LAYER} ...")

X_detect = H[:, DETECT_LAYER, :]
probe    = LogisticRegression(max_iter=1000, C=1.0, random_state=RANDOM_SEED)
probe.fit(X_detect, y)

# Probe confidence = P(factual) at layer 13
probe_conf = probe.predict_proba(X_detect)[:, 1]   # [N], higher = more factual

# Quick AUROC check
from sklearn.metrics import roc_auc_score
auroc = roc_auc_score(y, probe_conf)
print(f"Probe AUROC at L*={DETECT_LAYER}: {auroc:.4f} ✓")

torch.save({
    'probe_coef' : probe.coef_,
    'probe_inter': probe.intercept_,
    'probe_conf' : probe_conf
}, f'{SWEEP_DIR}/probe_l{DETECT_LAYER}.pt')

In [ ]:
def steered_generate(model, tokenizer, input_ids, steer_layer, steer_vec, alpha, max_new_tokens=15):
    device    = input_ids.device
    steer_vec = steer_vec.to(device).half()
    generated = input_ids.clone()

    for _ in range(max_new_tokens):
        with torch.no_grad():
            h            = model.model.embed_tokens(generated)
            seq_len      = h.shape[1]
            position_ids = torch.arange(seq_len, device=device).unsqueeze(0)

            # Build causal attention mask
            causal_mask = torch.full(
                (1, 1, seq_len, seq_len), float('-inf'),
                device=device, dtype=h.dtype
            )
            causal_mask = torch.triu(causal_mask, diagonal=1)

            # Precompute RoPE
            cos, sin = model.model.rotary_emb(h, position_ids)

            for layer_idx, layer in enumerate(model.model.layers):
                if layer_idx == steer_layer:
                    h = h + alpha * steer_vec.unsqueeze(0).unsqueeze(0)

                # Call attention and MLP separately to avoid position_embeddings issue
                residual = h
                h = layer.input_layernorm(h)

                # Self attention with explicit RoPE
                attn_out = layer.self_attn(
                    hidden_states=h,
                    attention_mask=causal_mask,
                    position_ids=position_ids,
                    position_embeddings=(cos, sin),
                    past_key_value=None,
                    output_attentions=False,
                    use_cache=False,
                )
                h = residual + attn_out[0]

                # MLP
                residual = h
                h = layer.post_attention_layernorm(h)
                h = layer.mlp(h)
                h = residual + h

            h        = model.model.norm(h)
            logits   = model.lm_head(h)
            next_tok = logits[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_tok], dim=1)

            if next_tok.item() == tokenizer.eos_token_id:
                break

    return generated

In [ ]:
# ── CELL 7: Two-panel figure — Detection vs Correction ───────────
# LEFT PANEL  → Probe AUROC by layer   (detection)
# RIGHT PANEL → Steering accuracy by layer (correction)
# Both panels have key method layers marked as dots.

print("Generating two-panel figure ...")

# Load Q4 AUROC curve
auroc_path = f'{SAVE_DIR}/layer_aurocs.npy'
assert os.path.exists(auroc_path), "Run Q4 notebook first to get layer_aurocs.npy"
layer_aurocs = np.load(auroc_path)   # [L+1]

# Steering acc curve (layer index 0 = layer 1 in model)
# Pad with baseline at index 0 (embedding layer, no steering)
steer_accs_full = np.concatenate([[baseline_acc], layer_accs])

depth_detect = np.arange(len(layer_aurocs))  / NUM_LAYERS * 100
depth_steer  = np.arange(len(steer_accs_full)) / NUM_LAYERS * 100

CORRECT_LAYER = int(np.argmax(layer_accs)) + 1   # expected: 24

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('#0D1117')
for ax in [ax1, ax2]:
    ax.set_facecolor('#0D1117')
    ax.tick_params(colors='#A0AEC0', labelsize=10)
    for spine in ax.spines.values():
        spine.set_edgecolor('#2D3748')
    ax.grid(True, color='#2D3748', linestyle='--',
            linewidth=0.5, alpha=0.4)

In [ ]:
# ── LEFT: Detection (AUROC) ──
ax1.plot(depth_detect, layer_aurocs,
         color='#63B3ED', linewidth=2, alpha=0.9, zorder=2)
ax1.fill_between(depth_detect, 0.5, layer_aurocs,
                 color='#63B3ED', alpha=0.08)
ax1.axhline(0.5, color='#4A5568', linestyle='--',
            linewidth=1, alpha=0.6)

# Mark L*=13
detect_depth = DETECT_LAYER / NUM_LAYERS * 100
detect_auroc = layer_aurocs[DETECT_LAYER]
ax1.scatter(detect_depth, detect_auroc,
            color='#38A169', marker='*', s=300,
            zorder=5, edgecolors='white', linewidths=1.5,
            label=f'Our L*={DETECT_LAYER} (AUROC={detect_auroc:.3f})')
ax1.plot([detect_depth, detect_depth], [0.5, detect_auroc],
         color='#38A169', linestyle=':', linewidth=1.5, alpha=0.6)

# Mark SAPLMA (last layer)
saplma_depth = NUM_LAYERS / NUM_LAYERS * 100
saplma_auroc = layer_aurocs[NUM_LAYERS]
ax1.scatter(saplma_depth, saplma_auroc,
            color='#E53E3E', marker='v', s=160,
            zorder=5, edgecolors='white', linewidths=1.5,
            label=f'SAPLMA L={NUM_LAYERS} (AUROC={saplma_auroc:.3f})')

ax1.set_xlabel('Layer Depth (%)', fontsize=12,
               color='#E2E8F0', labelpad=8)
ax1.set_ylabel('Probe AUROC (5-fold CV)', fontsize=12,
               color='#E2E8F0', labelpad=8)
ax1.set_title('Detection: Probe AUROC by Layer',
              fontsize=13, color='white', fontweight='bold', pad=12)
ax1.set_xlim(-2, 107)
ax1.set_ylim(0.45, min(1.02, layer_aurocs.max() + 0.06))
ax1.legend(loc='lower right', fontsize=9,
           facecolor='#1A202C', edgecolor='#2D3748',
           labelcolor='#E2E8F0', framealpha=0.9)

In [ ]:
# ── RIGHT: Correction (Steering Accuracy) ──
ax2.plot(depth_steer, steer_accs_full,
         color='#F6AD55', linewidth=2, alpha=0.9, zorder=2)
ax2.fill_between(depth_steer,
                 baseline_acc, steer_accs_full,
                 where=steer_accs_full >= baseline_acc,
                 color='#38A169', alpha=0.12,
                 label='Above baseline')
ax2.fill_between(depth_steer,
                 baseline_acc, steer_accs_full,
                 where=steer_accs_full < baseline_acc,
                 color='#E53E3E', alpha=0.12,
                 label='Below baseline')
ax2.axhline(baseline_acc, color='#4A5568', linestyle='--',
            linewidth=1.2, alpha=0.8,
            label=f'Baseline (no steering): {baseline_acc:.3f}')

# Mark L_correct
correct_depth = CORRECT_LAYER / NUM_LAYERS * 100
correct_acc   = steer_accs_full[CORRECT_LAYER]
ax2.scatter(correct_depth, correct_acc,
            color='#38A169', marker='*', s=300,
            zorder=5, edgecolors='white', linewidths=1.5,
            label=f'L_correct={CORRECT_LAYER} (acc={correct_acc:.3f})')
ax2.plot([correct_depth, correct_depth],
         [baseline_acc, correct_acc],
         color='#38A169', linestyle=':', linewidth=1.5, alpha=0.6)

# Mark L*=13 on steering panel — to show it's NOT optimal for steering
detect_steer_acc = steer_accs_full[DETECT_LAYER]
ax2.scatter(detect_depth, detect_steer_acc,
            color='#E53E3E', marker='v', s=160,
            zorder=5, edgecolors='white', linewidths=1.5,
            label=f'L*={DETECT_LAYER} steered (acc={detect_steer_acc:.3f})')

# Annotate the gap
ax2.annotate('',
    xy=(correct_depth, correct_acc + 0.005),
    xytext=(detect_depth, correct_acc + 0.005),
    arrowprops=dict(arrowstyle='<->', color='#CBD5E0',
                    lw=1.5))
ax2.text((detect_depth + correct_depth) / 2,
         correct_acc + 0.010,
         f'Detect-Correct Gap\n({CORRECT_LAYER - DETECT_LAYER} layers, '
         f'{(CORRECT_LAYER - DETECT_LAYER)/NUM_LAYERS*100:.0f}%)',
         ha='center', fontsize=8, color='#CBD5E0')

ax2.set_xlabel('Layer Depth (%)', fontsize=12,
               color='#E2E8F0', labelpad=8)
ax2.set_ylabel('Factual Accuracy (CAA steered)', fontsize=12,
               color='#E2E8F0', labelpad=8)
ax2.set_title('Correction: Steering Accuracy by Layer',
              fontsize=13, color='white', fontweight='bold', pad=12)
ax2.set_xlim(-2, 107)
ax2.legend(loc='lower right', fontsize=9,
           facecolor='#1A202C', edgecolor='#2D3748',
           labelcolor='#E2E8F0', framealpha=0.9)

fig.suptitle(
    'The Detect-Correct Discrepancy: Hallucination Detection and Correction\n'
    'Require Different Layers  —  Llama-3.2-3B-Instruct | TriviaQA (n=500)',
    fontsize=14, color='white', fontweight='bold', y=1.02
)

plt.tight_layout()
fig_path = f'{SWEEP_DIR}/detect_correct_discrepancy.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()
print(f"Figure saved → {fig_path}")

In [ ]:
# ── CELL 8: PART 2 — Adaptive per-sample steering ────────────────
# Router logic:
#   - Get probe confidence at L*=13: P(factual | h_13)
#   - High confidence factual (conf > HIGH_THRESH) → no steering
#   - High confidence hallucinated (conf < LOW_THRESH) → full alpha
#   - Borderline → partial alpha (scaled by uncertainty)
#
# This is the novel contribution on top of the sweep finding.
# Compare three conditions:
#   1. No steering (baseline)
#   2. Fixed CAA at L_correct for all samples
#   3. Adaptive routing — steer magnitude scales with probe confidence

HIGH_THRESH = 0.65   # above this → likely factual → don't steer
LOW_THRESH  = 0.35   # below this → likely hallucinated → full steer
ALPHA_MAX   = 2.0    # maximum steering magnitude
ALPHA_MIN   = 0.0    # minimum (no steering)

print("\n" + "="*60)
print("PART 2: ADAPTIVE PER-SAMPLE STEERING")
print("="*60)
print(f"  Detection layer : L*={DETECT_LAYER}")
print(f"  Correction layer: L_correct={CORRECT_LAYER}")
print(f"  High threshold  : {HIGH_THRESH} (factual → skip)")
print(f"  Low threshold   : {LOW_THRESH}  (hallucinated → full steer)")
print(f"  Alpha range     : [{ALPHA_MIN}, {ALPHA_MAX}]")

steering_vec_correct = steering_vecs[CORRECT_LAYER]

def adaptive_alpha(conf):
    """
    Given probe confidence P(factual):
      conf > HIGH_THRESH → alpha = 0 (don't touch)
      conf < LOW_THRESH  → alpha = ALPHA_MAX (full steer)
      between thresholds → linearly interpolate
    """
    if conf > HIGH_THRESH:
        return 0.0
    elif conf < LOW_THRESH:
        return ALPHA_MAX
    else:
        # Linear interpolation: higher conf = lower alpha
        t = (HIGH_THRESH - conf) / (HIGH_THRESH - LOW_THRESH)
        return ALPHA_MIN + t * (ALPHA_MAX - ALPHA_MIN)

ADAPTIVE_CACHE = f'{SWEEP_DIR}/adaptive_results.pt'

if os.path.exists(ADAPTIVE_CACHE):
    print("Adaptive cache found — loading ...")
    adaptive_results = torch.load(ADAPTIVE_CACHE)
else:
    results_fixed    = {'correct': 0, 'steered': 0, 'skipped': 0}
    results_adaptive = {'correct': 0, 'steered': 0, 'skipped': 0}

    for i, item in enumerate(tqdm(samples, desc="Adaptive steering")):
        question = item['question']
        gold_ans = item['answer']['aliases'] + [item['answer']['value']]
        prompt   = PROMPT_TMPL.format(q=question)
        inputs   = tokenizer(prompt, return_tensors='pt').to(DEVICE)
        conf     = float(probe_conf[i])

        # ── Fixed CAA at L_correct ──
        alpha_fixed = ALPHA_MAX

        def make_hook_alpha(vec, alpha):
            def hook_fn(module, input, output):
                if alpha == 0.0:
                    return output
                hidden = output[0]
                hidden = hidden + alpha * vec.unsqueeze(0).unsqueeze(0)
                return (hidden,) + output[1:]
            return hook_fn

        hook_fixed = model.model.layers[CORRECT_LAYER - 1].register_forward_hook(
            make_hook_alpha(steering_vec_correct, alpha_fixed)
        )
        try:
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOK,
                                     do_sample=False)
            pred = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                    skip_special_tokens=True).strip()
            if token_f1(pred, gold_ans) > 0.5:
                results_fixed['correct'] += 1
            results_fixed['steered'] += 1
        finally:
            hook_fixed.remove()

        # ── Adaptive steering ──
        alpha_adapt = adaptive_alpha(conf)

        hook_adapt = model.model.layers[CORRECT_LAYER - 1].register_forward_hook(
            make_hook_alpha(steering_vec_correct, alpha_adapt)
        )
        try:
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOK,
                                     do_sample=False)
            pred = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                    skip_special_tokens=True).strip()
            if token_f1(pred, gold_ans) > 0.5:
                results_adaptive['correct'] += 1
            if alpha_adapt == 0.0:
                results_adaptive['skipped'] += 1
            else:
                results_adaptive['steered'] += 1
        finally:
            hook_adapt.remove()

        del out, inputs
        if i % 20 == 0:
            gc.collect()
            torch.cuda.empty_cache()

    adaptive_results = {
        'baseline_acc' : baseline_acc,
        'fixed_acc'    : results_fixed['correct']    / NUM_SAMPLES,
        'adaptive_acc' : results_adaptive['correct'] / NUM_SAMPLES,
        'fixed'        : results_fixed,
        'adaptive'     : results_adaptive,
    }
    torch.save(adaptive_results, ADAPTIVE_CACHE)

print("\n" + "="*60)
print("ADAPTIVE STEERING RESULTS")
print("="*60)
print(f"  Baseline (no steering)   : {adaptive_results['baseline_acc']*100:.2f}%")
print(f"  Fixed CAA (L={CORRECT_LAYER})       : {adaptive_results['fixed_acc']*100:.2f}%  "
      f"(+{(adaptive_results['fixed_acc']-adaptive_results['baseline_acc'])*100:.2f}pp)")
print(f"  Adaptive routing         : {adaptive_results['adaptive_acc']*100:.2f}%  "
      f"(+{(adaptive_results['adaptive_acc']-adaptive_results['baseline_acc'])*100:.2f}pp)")
print(f"\n  Samples skipped (confident factual): "
      f"{adaptive_results['adaptive']['skipped']}/{NUM_SAMPLES}")
print("="*60)

In [ ]:
# ── CELL 9: Adaptive results bar chart ───────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor('#0D1117')
ax.set_facecolor('#0D1117')

methods  = ['Baseline\n(no steering)',
            f'Fixed CAA\n(L={CORRECT_LAYER})',
            'Adaptive\nRouting (Ours)']
accs     = [adaptive_results['baseline_acc'],
            adaptive_results['fixed_acc'],
            adaptive_results['adaptive_acc']]
colors   = ['#4A5568', '#ED8936', '#38A169']

bars = ax.bar(methods, [a * 100 for a in accs],
              color=colors, width=0.5,
              edgecolor='white', linewidth=0.8)

# Value labels on bars
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{acc*100:.2f}%',
            ha='center', va='bottom',
            color='white', fontsize=11, fontweight='bold')

# Delta annotations
for i in range(1, 3):
    delta = (accs[i] - accs[0]) * 100
    ax.text(i, accs[i] * 100 / 2,
            f'+{delta:.1f}pp',
            ha='center', va='center',
            color='white', fontsize=9, alpha=0.8)

ax.set_ylabel('Factual Accuracy (%)', fontsize=12,
              color='#E2E8F0', labelpad=8)
ax.set_title('Adaptive Per-Sample Steering Outperforms Fixed CAA\n'
             f'Llama-3.2-3B | TriviaQA | L*={DETECT_LAYER} → L_correct={CORRECT_LAYER}',
             fontsize=12, color='white', fontweight='bold', pad=12)
ax.tick_params(colors='#A0AEC0', labelsize=10)
for spine in ax.spines.values():
    spine.set_edgecolor('#2D3748')
ax.set_ylim(0, max(accs) * 100 * 1.12)
ax.grid(True, axis='y', color='#2D3748',
        linestyle='--', linewidth=0.5, alpha=0.4)

plt.tight_layout()
fig2_path = f'{SWEEP_DIR}/adaptive_steering_results.png'
plt.savefig(fig2_path, dpi=150, bbox_inches='tight',
            facecolor='#0D1117')
plt.show()
print(f"Figure saved → {fig2_path}")
print(f"\nAll outputs saved to: {SWEEP_DIR}")